# Map Showcase to StoryMap Pipeline

### Imports

In [ ]:
from arcgis.apps.storymap import StoryMap, story, story_content
from arcgis.apps.storymap.story_content import Image, TextStyles, Video, Audio, Embed, Map, Cover, Text, Button, Gallery, Swipe, Sidecar, Timeline, SidecarSlide, Scales
from arcgis.apps.storymap.collection import Collection 
import arcgis.geometry as arcGeo
import time
import getpass
from arcgis.gis import GIS
from functools import wraps
import random
import requests

### Dev Portal Login

In [ ]:
while True:
    portal_url=input("Please input the URL to your dev organization:")
    portal_username=input("Please input the username to your dev organization:")
    portal_password=getpass.getpass(prompt="Enter the password to your dev organization:")
    try:
        gis = GIS(
                url=portal_url, # portal URL
                username=portal_username, # portal username
                password=portal_password # portal password
            )
        if gis:
            print("Succesful login!")
            break # breaking loop with successful login
    except Exception as e:
        print(f"Unable to login to with the credentials provided with the following error: {e}. Please try again.\n")

### State Abbrevations Dictionary

In [ ]:
# dictionary to map state names to two letter abbreviations, used for updating the StoryMap collection's thumbnail with the state flag
states_dict = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
}

### Backoff logic decorator

In [ ]:
def exponential_backoff(max_retries=5, base_delay=1, max_delay=60):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            delay=base_delay
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries - 1:
                        raise e # failing after last retry
                    # calculating exponential backoff wait
                    sleep_time = min(delay, max_delay)
                    print(f"Attempt {attempt + 1} failed: {e}. Retrying in {sleep_time}s...")
                    time.sleep(sleep_time)
                    delay *= 2.0  # Doubling base delay for the next round
        return wrapper
    return decorator

### Creating StoryMaps from a Showcase's web maps

In [ ]:
@exponential_backoff(max_retries=4, base_delay=2.0)
def updateItemDetails(item, new_thumbnail_path, new_description, new_summary):
    return item.update(
        item_properties={
            "access": "public",
            "description": new_description,
            "snippet": new_summary,
        },
        thumbnail=new_thumbnail_path
    )
    
class webMap:
    """
        A class representing a Map stored in a Map Showcase

    Attributes:
        wm_itemId (str): The underlying AGOL item ID associated with the Map 
        wm_title (str): The title of the Map within the showcase.
        wm_description (str): The description of the Map within the showcase.
        wm_summary (str): The summary (aka snippet) of the Map within the showcase.
        wm_geometry (list): The geometry of the Highlight Area for a Map within the showcase.
        wm_latestWkid (list): The latest well-known ID of the Highlight Area geometry for a Map within the showcase.
        wm_wkid (list): The well-known ID of the Highlight Area geometry for a Map within the showcase.
    """

    def __init__(self, wm_itemId, wm_title, wm_description, wm_summary, wm_geometry, wm_latestWkid, wm_wkid):
        
        # gathering details from the item
        self.itemId = wm_itemId
        self.item = gis.content.get(self.itemId)
        self.title = wm_title
        self.thumbnail_path = self.item.download_thumbnail() # retrieves the link to the thumbnail
        self.thumbnail = Image(self.thumbnail_path)
        self.description = wm_description
        self.summary = wm_summary
        self.geometry = wm_geometry
        self.geometryExtent = self.geometry.extent
        self.latestWkid = wm_latestWkid
        self.wkid = wm_wkid
        
def bufferMaskExtent(extent, input_latest_wkid, input_wkid, buffer_ratio=0.25):
    """Buffer an extent, defaulting to 25%."""
    width = extent[2] - extent[0]  # xmax - xmin
    height = extent[3] - extent[1]  # ymax - ymin
    
    new_extent = {
        'xmin': extent[0] - (width * buffer_ratio),
        'ymin': extent[1] - (height * buffer_ratio),
        'xmax': extent[2] + (width * buffer_ratio),
        'ymax': extent[3] + (height * buffer_ratio),
        'spatialReference': {'latestWkid': input_latest_wkid, 'wkid': input_wkid},
    }
    return new_extent
    
def createStoryMapforWebMap(webMapObject):
    """
        Creating a StoryMap for one of the Maps within a Map Showcase. StoryMap contains a sidecar and topical map.

    Parameters:
        webMapObject (object): The custom class Web Map object created for the Map within the Map Showcase.

    Returns:
        dictionary: Information about the StoryMap created, which will be used to populate a StoryMap collection.
    """
    
    # creating an empty storymap
    new_story = StoryMap()

    # filling in the header & byline
    new_story.contents[0].title = webMapObject.title
    new_story.contents[0].summary = 'Made with the ArcGIS API for Python'
    new_story.contents[0].date = 'current-date'

    buffered_extent = bufferMaskExtent(webMapObject.geometry.extent, webMapObject.latestWkid, webMapObject.wkid)

    # adding narrative content
    sidecar_heading = Text(webMapObject.title, style=TextStyles.HEADING)
    sidecar_paragraph = Text("This is a test paragraph, included for the API.")
    button = Button(
        link= f'{portal_url}/home/item.html?id={webMapObject.itemId}#overview',
        text="Original Web Map")
    narrative_panel = [sidecar_heading, sidecar_paragraph, button]

    # creating a map for the sidecar
    sidecar_map = Map(webMapObject.itemId)
    sidecar_map.set_viewpoint(extent=buffered_extent, scale=Scales.STATES)

    # assembling the sidecar and adding it to the storymap
    slide = SidecarSlide(content=narrative_panel,media=sidecar_map)
    sidecar = Sidecar(style="docked-panel")
    sidecar.slides = [slide]
    new_story.add(sidecar)
   
    # adding credits
    new_story.credits("StoryMaps" , "Python API", "Thank You for Reading","Esri Living Atlas of the World")
    
    # saving, returns the AGOL item ID of the StoryMap
    sm_item = new_story.save(title=webMapObject.title) 
    
    # updating the StoryMap's item details
    updateItemDetails(
        item=sm_item,
        new_thumbnail_path=webMapObject.thumbnail_path,
        new_description=webMapObject.description,
        new_summary=webMapObject.summary,
    )    
    
    # returning a dictionary of information on the StoryMap to then use it in a collection
    sm_dict = {
        'storymap': new_story,
        'storymap_item': sm_item,
        'thumbnail_path': webMapObject.thumbnail_path,
        'thumbnail': webMapObject.thumbnail,
    }
    return sm_dict
    
def createCollectionFromShowcase(inputID):
    """
        Main workhorse function to create a StoryMap Collection from the constituent maps of a Map Showcase.

    Parameters:
        inputID (string): The AGOL item ID of the Map Showcase.

    Returns:
        object: The StoryMap collection created.
    """
    
    #-------------------- PARSING THE MAP SHOWCASE --------------------
    showcase = gis.content.get(inputID)
    
    all_showcase_data = showcase.get_data()

    # a dictionary to hold information about each of the Maps within the Map Showcase
    all_storymaps_from_showcase = {}

    # the Higlighted Area of the Maps within the Showcase. For our purposes, this is the same across all Maps in the Map Showcase
    focus_geography = None

    # looping through the Maps in the Map Showcase
    print("Creating showcase for item:", inputID)
    for item in all_showcase_data['items']:
        
        if item['type'] == "Web Map":

            # the AGOL ID of the item
            item_id = item['id']
            
            # showcase override details
            title = item['title']
            item_description = item['description']
            summary = item['snippet']
            
            # creating a union of the geometries 
            all_geometries = []
            
            # these will get reassigned in the loop, but they shouldn't change between geometries
            latestWkid = None
            wkid = None
            
            geography_names = []
            for geometry in item['highlightedArea']['graphics']:
                srs = geometry['geometry']['spatialReference']
                current_geometry = arcGeo.Geometry(geometry['geometry'])
                all_geometries.append(current_geometry)
                latestWkid = geometry['geometry']['spatialReference']['latestWkid']
                wkid = geometry['geometry']['spatialReference']['wkid']
                geography_names.append(geometry['attributes']['NAME'])
            
            geometries_union = arcGeo.union(all_geometries, spatial_ref = srs)[0] # unionizing the geometries
            geography_names = list(set(geography_names)) # unique geographies, ideally this should be of length 1
            focus_geography = geography_names[0] # taking the first geography's NAME
    
            print(f'Creating WebMap for item id: {item_id}')
            currentWebMap = webMap(item_id, title, item_description, summary, geometries_union, latestWkid, wkid) # creating a webmap for the item
            
            print(f'Creating StoryMap for item id: {currentWebMap.title}')
            currentStoryMapDict = createStoryMapforWebMap(currentWebMap) # creating a StoryMap for the current Map's data
            all_storymaps_from_showcase[item_id] = currentStoryMapDict # adding the StoryMap to a dictionary, which will be combined later into a StoryMap collection 
    
    #-------------------- CREATING A COLLECTION --------------------
    print("Creating a StoryMap collection")
    coll = Collection()
    coll.title = f'{focus_geography} Map Portfolio' 
    coll.content[0].title = f'{focus_geography} Map Portfolio' 
    for sm_item, storymap_dict in all_storymaps_from_showcase.items():
        coll.add(item=storymap_dict['storymap_item'], thumbnail=storymap_dict['thumbnail_path']) # the add method expects a thumbnail string
        # coll.add(item=storymap_dict['storymap_item']) # commenting out the thumbnail to avoid issues
    collectionTitle = f"{showcase.title.replace("Showcase", "StoryMap Collection")}"
    coll_item = coll.save(title=collectionTitle, access='public')

    #-------------------- UPDATING THE COLLECTION DETAILS --------------------
    print("Updating the collection details")
    # converting the state name to the 2-letter abbreviation
    state_abbr = states_dict[focus_geography]
    updateItemDetails(
        item=coll_item,
        new_thumbnail_path=f"https://github.com/newtdobbs/State_Flags_png/blob/main/{states_dict[focus_geography].lower()}.png?raw=true",
        new_summary='State map portfolio collection',
        new_description=f'This collection of {focus_geography} maps enables decision-makers to monitor current conditions in the state across multiple topics.',
    )

    return coll_item

### Calling functions

In [ ]:
showcaseID = input('Paste in an item ID for a Showcase item.')
createCollectionFromShowcase(showcaseID)